# LargestLLM Router - Inference

This notebook demonstrates the **LargestLLM** baseline router.

## Overview

LargestLLM is a simple baseline router that always routes queries to the largest model in the candidate pool.
This serves as an upper bound for routing performance and a lower bound for cost efficiency.

**Key Characteristics**:
- No training required (deterministic baseline)
- Always selects the largest model by parameter size
- Useful for performance benchmarking
- Highest expected quality, highest cost

## 1. Environment Setup

In [1]:
# For Google Colab: Clone repository and install dependencies
import os

!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install pyyaml transformers torch

In [15]:
from llmrouter.models.largest_llm import LargestLLM
from llmrouter.utils import setup_environment
import yaml

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

LargestLLM router requires only data paths - no hyperparameters.

| Parameter | Description |
|-----------|-------------|
| `llm_data` | Path to LLM candidate metadata |
| `routing_data_test` | Path to test routing data |

In [16]:
CONFIG_PATH = "configs/model_config_test/largest_llm.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  query_embedding_data: data/example_data/routing_data/query_embeddings_longformer.pt
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
metric:
  weights:
    cost: 0
    llm_judge: 0
    performance: 1



## 3. Initialize Router

In [17]:
router = LargestLLM(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of LLM candidates: {len(router.llm_data)}")

✅ MetaRouter initialized successfully (YAML + data loaded).
✅ LargestLLM initialized successfully
Router initialized successfully!
Number of LLM candidates: 7


In [18]:
# Display available LLM candidates sorted by size
print("Available LLM Candidates (by size):")
print("=" * 60)

llm_list = [(name, info.get('size', 'unknown')) for name, info in router.llm_data.items()]
llm_list_sorted = sorted(llm_list, key=lambda x: float(x[1]) if isinstance(x[1], (int, float)) else 0, reverse=True)

for i, (name, size) in enumerate(llm_list_sorted, 1):
    marker = " <- LARGEST" if i == 1 else ""
    print(f"{i}. {name}: {size}B parameters{marker}")

Available LLM Candidates (by size):
1. qwen2.5-7b-instruct: 7BB parameters <- LARGEST
2. llama-3.1-8b-instruct: 8BB parameters
3. mistral-7b-instruct-v0.3: 7BB parameters
4. llama-3.3-nemotron-super-49b-v1: 49BB parameters
5. llama3-70b-instruct: 70BB parameters
6. mixtral-8x7b-instruct-v0.1: 45BB parameters
7. mixtral-8x22b-instruct-v0.1: 141BB parameters


## 4. Query Routing

LargestLLM always routes to the largest model, regardless of query complexity.

In [25]:
EXAMPLE_QUERIES = [
    {"query": "What is 2 + 2?"},  # Simple
    {"query": "Explain the theory of general relativity."},  # Medium
    {"query": "Prove P != NP."},  # Complex
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")
    print()

Routing Results:
1. What is 2 + 2?...
   Routed to: mixtral-8x22b-instruct-v0.1

2. Explain the theory of general relativity....
   Routed to: mixtral-8x22b-instruct-v0.1

3. Prove P != NP....
   Routed to: mixtral-8x22b-instruct-v0.1



## 5. Batch Routing

In [32]:
# Route test data
test_queries = router.routing_data_test[:10]

print(f"Routing {len(test_queries)} test queries...")
results = router.route_batch(test_queries.to_dict('records'))

print(f"\nRouting Distribution:")
from collections import Counter
model_counts = Counter([r['model_name'] for r in results])
for model, count in model_counts.most_common():
    print(f"  {model}: {count} ({100*count/len(results):.1f}%)")

Routing 10 test queries...

Routing Distribution:
  mixtral-8x22b-instruct-v0.1: 10 (100.0%)


## 6. Evaluation

In [66]:
from llmrouter.evaluation import evaluate_batch
from collections import Counter

eval_data = []
for r in results:
    if r.get('ground_truth'):
        eval_data.append({
            'prediction': r.get('response', ''),
            'ground_truth': r.get('ground_truth'),
            'metric': r.get('metric', 'em')
        })

if eval_data:
    eval_results = evaluate_batch(eval_data)
    scores = [r['score'] for r in eval_results]
    avg_score = sum(scores) / len(scores) if scores else 0
else:
    eval_results = []
    avg_score = 0

largest_metrics = {
    'avg_performance': sum(r.get('score', 0) for r in eval_results) / len(eval_results) if eval_results else 0,
    'total_queries': len(eval_results),
    'success_count': sum(1 for r in eval_results if r.get('score', 0) > 0)
}

routing_dist = Counter(r['model_name'] for r in results)

print("\nEvaluation Metrics:")
print("-" * 50)
print(f"Total queries: {len(results)}")
print(f"Evaluated queries: {len(eval_data)}")
print(f"Average score ({eval_data[0]['metric'] if eval_data else 'N/A'}): {avg_score:.4f}")

if eval_results:
    print("\nDetailed Scores:")
    print("-" * 50)
    for i, r in enumerate(eval_results[:5], 1):
        pred = r.get('prediction', '')[:30]
        gt = r.get('ground_truth', '')[:30]
        print(f"  {i}. Score: {r['score']:.4f} | pred: {pred} | ground truth: {gt}")



Evaluation Metrics:
--------------------------------------------------
Total queries: 10
Evaluated queries: 10
Average score (em_mc): 0.0000

Detailed Scores:
--------------------------------------------------
  1. Score: 0.0000 | pred:  Let's analyze the clues step  | ground truth: C
  2. Score: 0.0000 | pred:  From the clues, we can deduce | ground truth: A
  3. Score: 0.0000 | pred:  The person who likes yellow l | ground truth: A
  4. Score: 0.0000 | pred:  To solve this problem, we nee | ground truth: D
  5. Score: 0.0000 | pred:  To solve this problem, we nee | ground truth: D


## 7. Compare with SmallestLLM

Compare the two baseline routers to understand the performance-cost tradeoff.

In [70]:
from llmrouter.evaluation import evaluate_batch
from llmrouter.models.smallest_llm import SmallestLLM

smallest_router = SmallestLLM(yaml_path="configs/model_config_test/smallest_llm.yaml")
smallest_test_queries = router.routing_data_test[:10]
smallest_results = smallest_router.route_batch(smallest_test_queries.to_dict('records'))

eval_data = [{
    'prediction': r.get('response', ''),
    'ground_truth': r.get('ground_truth', ''),
    'metric': r.get('metric', 'em'),
    'task_name': r.get('task_name', None)
} for r in smallest_results]

smallest_eval_results = evaluate_batch(eval_data)

smallest_metrics = {
    'avg_performance': sum(r.get('score', 0) for r in smallest_eval_results) / len(smallest_eval_results) if smallest_eval_results else 0,
    'total_queries': len(smallest_eval_results),
    'success_count': sum(1 for r in smallest_eval_results if r.get('score', 0) > 0)
}

print("Comparison: SmallestLLM vs LargestLLM")
print("=" * 60)
print(f"{'Metric':<20} {'SmallestLLM':<15} {'LargestLLM':<15}")
print("-" * 60)
print(f"{'avg_performance':<20} {smallest_metrics['avg_performance']:<15.4f} {largest_metrics['avg_performance']:<15.4f}")
print(f"{'total_queries':<20} {smallest_metrics['total_queries']:<15} {largest_metrics['total_queries']:<15}")
print(f"{'success_count':<20} {smallest_metrics['success_count']:<15} {largest_metrics['success_count']:<15}")
print(f"{'success_rate':<20} {smallest_metrics['success_count']/smallest_metrics['total_queries']*100:<14.1f}% {largest_metrics['success_count']/largest_metrics['total_queries']*100:<14.1f}%")

✅ MetaRouter initialized successfully (YAML + data loaded).
✅ SmallestLLM initialized successfully

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Comparison: SmallestLLM vs LargestLLM
Metric               SmallestLLM     LargestLLM     
------------------------------------------------------------
avg_performance      0.0000          0.0000         
total_queries        10              10             
success_count        0               0              
success_rate         0.0           % 0.0           %


## 8. File-Based Inference

Load queries from a custom file and save results.

In [54]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from your own query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/largest_llm_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries from file
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries using route_batch
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results to file
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl
Routed 10 queries
Results saved to: outputs/largest_llm_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered... -> mixtral-8x22b-instruct-v0.1
  2. Q: There are 3 houses in a row, numbered... -> mixtral-8x22b-instruct-v0.1
  3. Q: There are 3 houses in a row, numbered... -> mixtral-8x22b-instruct-v0.1


## Summary

**LargestLLM Router**:
- Always routes to the largest model
- No training required (deterministic baseline)
- Provides upper bound for performance, lower bound for cost efficiency
- Useful for comparing against learned routing methods

**Use Cases**:
- Baseline comparison for routing research
- Quality-critical applications regardless of cost
- Establishing performance ceiling for routing methods